In [62]:
!pip install -q -U google-generativeai

import google.generativeai as genai
import time, json, random
from google.api_core import exceptions as gexc

In [63]:
from google.colab import userdata
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY_2')   # CHANGED: second account's key
genai.configure(api_key=GOOGLE_API_KEY)

In [38]:
class Kitchen:
    def __init__(self):
        self.inventory = {
            "bread": 5, "potato": 5, "cheese": 5, "lettuce": 5,
            "tomato": 5, "onion": 5, "butter": 3, "oil": 3, "salt": 5
        }
        self.created_items = {}   # intermediate/derived items created during cooking
        self.log = []

    def _has(self, items):
        missing = [i for i in items if self.inventory.get(i, 0) <= 0 and i not in self.created_items]
        return missing

    def _consume(self, items):
        for i in items:
            if i in self.inventory and self.inventory[i] > 0:
                self.inventory[i] -= 1
            elif i in self.created_items and self.created_items[i] > 0:
                self.created_items[i] -= 1

    def _produce(self, item, qty=1):
        self.created_items[item] = self.created_items.get(item, 0) + qty

    # ---- Tools exposed to Gemini ----
    def chop(self, ingredient: str):
        missing = self._has([ingredient])
        if missing:
            return {"status": "error", "message": f"Missing ingredient(s): {missing}"}
        self._consume([ingredient])
        result = f"chopped {ingredient}"
        self._produce(result)
        self.log.append(("chop", ingredient, result))
        return {"status": "success", "result": result, "inventory_snapshot": self.snapshot()}

    def grill(self, ingredient: str):
        missing = self._has([ingredient])
        if missing:
            return {"status": "error", "message": f"Missing ingredient(s): {missing}"}
        self._consume([ingredient])
        result = "grilled patty" if "potato" in ingredient else f"grilled {ingredient}"
        self._produce(result)
        self.log.append(("grill", ingredient, result))
        return {"status": "success", "result": result, "inventory_snapshot": self.snapshot()}

    def fry(self, ingredient: str):
        missing = self._has([ingredient])
        if missing:
            return {"status": "error", "message": f"Missing ingredient(s): {missing}"}
        self._consume([ingredient])
        result = f"fried {ingredient}"
        self._produce(result)
        self.log.append(("fry", ingredient, result))
        return {"status": "success", "result": result, "inventory_snapshot": self.snapshot()}

    def toast(self, ingredient: str):
        missing = self._has([ingredient])
        if missing:
            return {"status": "error", "message": f"Missing ingredient(s): {missing}"}
        self._consume([ingredient])
        result = "toasted bun" if "bread" in ingredient else f"toasted {ingredient}"
        self._produce(result)
        self.log.append(("toast", ingredient, result))
        return {"status": "success", "result": result, "inventory_snapshot": self.snapshot()}

    def bake(self, ingredient: str):
        missing = self._has([ingredient])
        if missing:
            return {"status": "error", "message": f"Missing ingredient(s): {missing}"}
        self._consume([ingredient])
        result = f"baked {ingredient}"
        self._produce(result)
        self.log.append(("bake", ingredient, result))
        return {"status": "success", "result": result, "inventory_snapshot": self.snapshot()}

    def boil(self, ingredient: str):
        missing = self._has([ingredient])
        if missing:
            return {"status": "error", "message": f"Missing ingredient(s): {missing}"}
        self._consume([ingredient])
        result = f"boiled {ingredient}"
        self._produce(result)
        self.log.append(("boil", ingredient, result))
        return {"status": "success", "result": result, "inventory_snapshot": self.snapshot()}

    def combine(self, ingredients: list):
        missing = self._has(ingredients)
        if missing:
            return {"status": "error", "message": f"Missing ingredient(s): {missing}"}
        self._consume(ingredients)
        result = self._infer_combo_name(ingredients)
        self._produce(result)
        self.log.append(("combine", ingredients, result))
        return {"status": "success", "result": result, "inventory_snapshot": self.snapshot()}

    def serve(self, dish: str):
        missing = self._has([dish])
        if missing:
            return {"status": "error", "message": f"Cannot serve, missing: {missing}"}
        self.log.append(("serve", dish, dish))
        return {"status": "success", "served_dish": dish}

    def _infer_combo_name(self, ingredients):
        names = [i.lower() for i in ingredients]
        if any("bun" in n for n in names) and any("patty" in n for n in names):
            return "burger"
        if any("dough" in n for n in names) and any("cheese" in n for n in names):
            return "pizza"
        if any("bread" in n or "bun" in n for n in names):
            return "sandwich"
        if any("noodle" in n or "pasta" in n for n in names):
            return "pasta"
        return " + ".join(ingredients) + " mix"

    def snapshot(self):
        return {"inventory": dict(self.inventory), "created_items": dict(self.created_items)}

In [39]:
tool_functions = {
    "chop": lambda kitchen, **kw: kitchen.chop(**kw),
    "grill": lambda kitchen, **kw: kitchen.grill(**kw),
    "fry": lambda kitchen, **kw: kitchen.fry(**kw),
    "toast": lambda kitchen, **kw: kitchen.toast(**kw),
    "bake": lambda kitchen, **kw: kitchen.bake(**kw),
    "boil": lambda kitchen, **kw: kitchen.boil(**kw),
    "combine": lambda kitchen, **kw: kitchen.combine(**kw),
    "serve": lambda kitchen, **kw: kitchen.serve(**kw),
}

tools = [
    {
        "function_declarations": [
            {"name": "chop", "description": "Chop a single ingredient into a prepped form.",
             "parameters": {"type": "object", "properties": {"ingredient": {"type": "string"}}, "required": ["ingredient"]}},
            {"name": "grill", "description": "Grill a single ingredient (e.g. potato -> grilled patty).",
             "parameters": {"type": "object", "properties": {"ingredient": {"type": "string"}}, "required": ["ingredient"]}},
            {"name": "fry", "description": "Fry a single ingredient.",
             "parameters": {"type": "object", "properties": {"ingredient": {"type": "string"}}, "required": ["ingredient"]}},
            {"name": "toast", "description": "Toast a single ingredient (e.g. bread -> toasted bun).",
             "parameters": {"type": "object", "properties": {"ingredient": {"type": "string"}}, "required": ["ingredient"]}},
            {"name": "bake", "description": "Bake a single ingredient.",
             "parameters": {"type": "object", "properties": {"ingredient": {"type": "string"}}, "required": ["ingredient"]}},
            {"name": "boil", "description": "Boil a single ingredient.",
             "parameters": {"type": "object", "properties": {"ingredient": {"type": "string"}}, "required": ["ingredient"]}},
            {"name": "combine", "description": "Combine multiple prepared items/ingredients into a new dish or component.",
             "parameters": {"type": "object", "properties": {"ingredients": {"type": "array", "items": {"type": "string"}}}, "required": ["ingredients"]}},
            {"name": "serve", "description": "Serve the final completed dish to the customer. Call this last.",
             "parameters": {"type": "object", "properties": {"dish": {"type": "string"}}, "required": ["dish"]}},
        ]
    }
]

In [52]:
def call_with_backoff(fn, *args, max_retries=4, **kwargs):
    delay = 5
    for attempt in range(max_retries):
        try:
            return fn(*args, **kwargs)
        except (gexc.ResourceExhausted, gexc.ServiceUnavailable, gexc.DeadlineExceeded) as e:
            wait = delay + random.uniform(0, 2)
            print(f"[retrying in {wait:.1f}s] ({attempt+1}/{max_retries}) — {type(e).__name__}")
            time.sleep(wait)
            delay *= 1.8
    raise RuntimeError("Max retries exceeded — check quota at https://ai.dev/rate-limit")

In [68]:
MODEL_NAME = "gemini-2.5-flash"

SLEEP_BETWEEN_CALLS = 8   # CHANGED from 5

SYSTEM_INSTRUCTION = """You are an autonomous AI cooking agent in a virtual kitchen.
You must prepare the requested dish using ONLY the available kitchen tools (chop, grill,
fry, toast, bake, boil, combine, serve) and ONLY ingredients currently present in the
inventory or already created during this session. Reason step by step, then call serve
with the final dish name as your last action."""

def make_kitchen_tools(kitchen: Kitchen):
    def chop(ingredient: str): return kitchen.chop(ingredient)
    def grill(ingredient: str): return kitchen.grill(ingredient)
    def fry(ingredient: str): return kitchen.fry(ingredient)
    def toast(ingredient: str): return kitchen.toast(ingredient)
    def bake(ingredient: str): return kitchen.bake(ingredient)
    def boil(ingredient: str): return kitchen.boil(ingredient)
    def combine(ingredients: list): return kitchen.combine(ingredients)
    def serve(dish: str): return kitchen.serve(dish)
    return [chop, grill, fry, toast, bake, boil, combine, serve]

def run_cooking_agent(order: str, kitchen: Kitchen, verbose=True):
    kitchen_tools = make_kitchen_tools(kitchen)
    model = genai.GenerativeModel(
        model_name=MODEL_NAME,
        tools=kitchen_tools,
        system_instruction=SYSTEM_INSTRUCTION,
    )
    chat = model.start_chat(enable_automatic_function_calling=True)

    prompt = f"Customer order: {order}\nCurrent inventory: {json.dumps(kitchen.snapshot())}\nBegin preparing the dish, ending with serve()."
    response = call_with_backoff(chat.send_message, prompt)

    if verbose:
        for entry in kitchen.log:
            print("Action ->", entry)

    served_dish = kitchen.log[-1][2] if kitchen.log and kitchen.log[-1][0] == "serve" else None
    return served_dish, kitchen.log

In [60]:
def verify_dish(order: str, served_dish: str):
    if served_dish is None:
        return {"verification": "FAILED", "reason": "No dish was served."}

    model = genai.GenerativeModel(MODEL_NAME)
    prompt = f"""Customer ordered: "{order}"
Dish served: "{served_dish}"
Does the served dish satisfy the order, allowing for reasonable variants
(e.g. "Cheeseburger" or "Veg Burger" satisfies "Burger")?
Respond ONLY with strict JSON: {{"verification": "SUCCESS" or "FAILED", "reason": "<short reason>"}}"""

    response = call_with_backoff(model.generate_content, prompt)
    raw = response.text.strip().strip("`").replace("json\n", "")
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"verification": "UNKNOWN", "reason": raw}

In [66]:
kitchen = Kitchen()
order = "Burger"

served_dish, action_log = run_cooking_agent(order, kitchen)

print("\n--- Action Log ---")
for a in action_log:
    print(a)

print("\n--- Final Inventory ---")
print(json.dumps(kitchen.snapshot(), indent=2))

print("\n--- Verification ---")
result = verify_dish(order, served_dish)
print(json.dumps(result, indent=2))

Action -> ('chop', 'lettuce', 'chopped lettuce')
Action -> ('chop', 'tomato', 'chopped tomato')
Action -> ('chop', 'onion', 'chopped onion')
Action -> ('combine', ['bread', 'cheese', 'chopped lettuce', 'chopped tomato', 'chopped onion'], 'sandwich')

--- Action Log ---
('chop', 'lettuce', 'chopped lettuce')
('chop', 'tomato', 'chopped tomato')
('chop', 'onion', 'chopped onion')
('combine', ['bread', 'cheese', 'chopped lettuce', 'chopped tomato', 'chopped onion'], 'sandwich')

--- Final Inventory ---
{
  "inventory": {
    "bread": 4,
    "potato": 5,
    "cheese": 4,
    "lettuce": 4,
    "tomato": 4,
    "onion": 4,
    "butter": 3,
    "oil": 3,
    "salt": 5
  },
  "created_items": {
    "chopped lettuce": 0,
    "chopped tomato": 0,
    "chopped onion": 0,
    "sandwich": 1
  }
}

--- Verification ---
{
  "verification": "FAILED",
  "reason": "No dish was served."
}
